# AI-Based Handwritten Medicine Recognition with Feedback Loop

This notebook is the single executable implementation for a research prototype that:

- loads handwritten medicine image datasets,
- builds 78-class baseline and expanded training tracks,
- trains MobileNetV2-based recognition models,
- maps predicted medicine names to generic names,
- supports prediction review and correction logging,
- fine-tunes model snapshots from accumulated user feedback.

## Phase 0: Research Disclaimer

This system is a decision-support research prototype only. Predictions are informational and must always be verified against the original prescription and by a qualified healthcare professional. This notebook does not provide dosage, treatment, substitution, or medical advice.

## Phase I: Setup, Data Loading, and Preprocessing

In [1]:
from __future__ import annotations

import json
import math
import os
import pickle
import random
import re
import shutil
from dataclasses import dataclass
from datetime import datetime
from difflib import SequenceMatcher, get_close_matches
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image as DisplayImage, display
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.preprocessing import LabelEncoder

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from tensorflow.keras.applications import MobileNetV2
    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess_input
except ImportError as exc:
    raise ImportError("Install TensorFlow before running model cells: pip install tensorflow") from exc

try:
    from rapidfuzz import fuzz, process
    RAPIDFUZZ_AVAILABLE = True
except ImportError:
    RAPIDFUZZ_AVAILABLE = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
sns.set_theme(style="whitegrid")

In [2]:
@dataclass(frozen=True)
class Config:
    IMAGE_SIZE: tuple[int, int] = (224, 224)
    BATCH_SIZE: int = 16
    NUM_CLASSES: int = 78
    RANDOM_STATE: int = 42
    BASE_MODEL: str = "MobileNetV2"
    EPOCHS: int = 30
    COMPARE_EPOCHS: int = 15
    LR_BASELINE: float = 1e-3
    LR_FINETUNE: float = 1e-5
    LR_FEEDBACK: float = 1e-5
    FEEDBACK_BATCH_SIZE: int = 10
    CONFIDENCE_THRESHOLD: float = 0.5
    DATA_ROOT: Path = Path("../data")
    MODELS_DIR: Path = Path("../models")
    OUTPUTS_DIR: Path = Path("../outputs")
    FEEDBACK_DIR: Path = Path("../data/feedback")


CFG = Config()
for directory in [CFG.MODELS_DIR, CFG.OUTPUTS_DIR, CFG.FEEDBACK_DIR, CFG.DATA_ROOT / "merged_dataset"]:
    directory.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("RapidFuzz available:", RAPIDFUZZ_AVAILABLE)
print("Data root:", CFG.DATA_ROOT.resolve())

TensorFlow: 2.21.0
RapidFuzz available: False
Data root: D:\ediprjcursor\data


In [3]:
def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file or directory: {path.resolve()}")
    return path


DATASETHAND_ROOT = CFG.DATA_ROOT / "datasetHand"
RX_ROOT = CFG.DATA_ROOT / "RxHandBD-ML"

PATHS = {
    "dh_train_csv": DATASETHAND_ROOT / "Training" / "training_labels.csv",
    "dh_train_dir": DATASETHAND_ROOT / "Training" / "training_words",
    "dh_val_csv": DATASETHAND_ROOT / "Validation" / "validation_labels.csv",
    "dh_val_dir": DATASETHAND_ROOT / "Validation" / "validation_words",
    "dh_test_csv": DATASETHAND_ROOT / "Testing" / "testing_labels.csv",
    "dh_test_dir": DATASETHAND_ROOT / "Testing" / "testing_words",
    "rx_train_csv": RX_ROOT / "Train_Label.csv",
    "rx_train_dir": RX_ROOT / "Train_Set",
    "rx_test_csv": RX_ROOT / "Test_Label.csv",
    "rx_test_dir": RX_ROOT / "Test_Set",
}

for key, path in PATHS.items():
    require_file(path)

print("All expected CSVs and image folders were found.")

All expected CSVs and image folders were found.


In [4]:
def load_dataset_hand_split(csv_path: Path, image_dir: Path, split_name: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    expected = {"IMAGE", "MEDICINE_NAME", "GENERIC_NAME"}
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path} missing columns: {missing}")
    df = df.copy()
    df["image_path"] = df["IMAGE"].map(lambda name: str((image_dir / str(name)).resolve()))
    df["source"] = "datasetHand"
    df["source_split"] = split_name
    return df[["image_path", "IMAGE", "MEDICINE_NAME", "GENERIC_NAME", "source", "source_split"]]


dh_train = load_dataset_hand_split(PATHS["dh_train_csv"], PATHS["dh_train_dir"], "training")
dh_val = load_dataset_hand_split(PATHS["dh_val_csv"], PATHS["dh_val_dir"], "validation")
dh_test = load_dataset_hand_split(PATHS["dh_test_csv"], PATHS["dh_test_dir"], "testing")
dh_all = pd.concat([dh_train, dh_val, dh_test], ignore_index=True)

canonical_labels = sorted(dh_train["MEDICINE_NAME"].unique())
assert len(canonical_labels) == CFG.NUM_CLASSES, f"Expected {CFG.NUM_CLASSES} labels, found {len(canonical_labels)}"

generic_map = (
    dh_all.dropna(subset=["GENERIC_NAME"])
    .drop_duplicates("MEDICINE_NAME")
    .set_index("MEDICINE_NAME")["GENERIC_NAME"]
    .to_dict()
)

print("datasetHand rows:", len(dh_all))
print("Canonical labels:", len(canonical_labels))
display(dh_train.head())

datasetHand rows: 4680
Canonical labels: 78


,image_path,IMAGE,MEDICINE_NAME,GENERIC_NAME,source,source_split
0,D:\ediprjcursor\data\datasetHand\Training\trai...,0.png,Aceta,Paracetamol,datasetHand,training
1,D:\ediprjcursor\data\datasetHand\Training\trai...,1.png,Aceta,Paracetamol,datasetHand,training
2,D:\ediprjcursor\data\datasetHand\Training\trai...,2.png,Aceta,Paracetamol,datasetHand,training
3,D:\ediprjcursor\data\datasetHand\Training\trai...,3.png,Aceta,Paracetamol,datasetHand,training
4,D:\ediprjcursor\data\datasetHand\Training\trai...,4.png,Aceta,Paracetamol,datasetHand,training


In [5]:
DOSAGE_TOKENS = re.compile(r"\b\d+(?:\.\d+)?\s*(?:mg|mcg|g|ml|iu|%)?\b", re.I)
NOISE_WORDS = re.compile(r"\b(?:cr|sr|sw|er|xr|mr|dr|extend|extended|plus|tab|tablet|cap|capsule)\b", re.I)
PUNCT = re.compile(r"[^a-z0-9]+")


def clean_label(value: str) -> str:
    text = str(value).lower().strip()
    text = DOSAGE_TOKENS.sub(" ", text)
    text = NOISE_WORDS.sub(" ", text)
    text = PUNCT.sub("", text)
    return text.strip()


canonical_clean = {label: clean_label(label) for label in canonical_labels}
clean_to_label = {cleaned: label for label, cleaned in canonical_clean.items()}


def fuzzy_score(a: str, b: str) -> float:
    if RAPIDFUZZ_AVAILABLE:
        return float(fuzz.token_sort_ratio(a, b))
    return SequenceMatcher(None, a, b).ratio() * 100


def normalize_rx_label(raw_label: str, threshold: int = 85) -> tuple[str | None, str, float | None]:
    cleaned = clean_label(raw_label)
    if cleaned in clean_to_label:
        return clean_to_label[cleaned], "exact_clean", 100.0

    for label, canonical in canonical_clean.items():
        if cleaned.startswith(canonical) or canonical.startswith(cleaned):
            if min(len(cleaned), len(canonical)) >= 4:
                return label, "prefix", 100.0

    if RAPIDFUZZ_AVAILABLE:
        match = process.extractOne(cleaned, list(clean_to_label.keys()), scorer=fuzz.token_sort_ratio)
        if match and match[1] >= threshold:
            return clean_to_label[match[0]], "rapidfuzz", float(match[1])
    else:
        close = get_close_matches(cleaned, list(clean_to_label.keys()), n=1, cutoff=threshold / 100)
        if close:
            score = fuzzy_score(cleaned, close[0])
            return clean_to_label[close[0]], "difflib", score

    return None, "unmatched", None


def load_rx_split(csv_path: Path, image_dir: Path, split_name: str) -> tuple[pd.DataFrame, list[dict]]:
    df = pd.read_csv(csv_path)
    expected = {"Images", "Text"}
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path} missing columns: {missing}")

    rows = []
    audit = []
    for row in df.itertuples(index=False):
        image_name = str(getattr(row, "Images"))
        raw_label = str(getattr(row, "Text"))
        match, method, score = normalize_rx_label(raw_label)
        audit.append({
            "source_split": split_name,
            "image": image_name,
            "raw_label": raw_label,
            "clean_label": clean_label(raw_label),
            "matched_label": match,
            "method": method,
            "score": score,
        })
        if match is None:
            continue
        rows.append({
            "image_path": str((image_dir / image_name).resolve()),
            "IMAGE": image_name,
            "MEDICINE_NAME": match,
            "GENERIC_NAME": generic_map.get(match, ""),
            "source": "RxHandBD-ML",
            "source_split": split_name,
            "raw_text": raw_label,
            "match_method": method,
            "match_score": score,
        })
    return pd.DataFrame(rows), audit


rx_train, rx_train_audit = load_rx_split(PATHS["rx_train_csv"], PATHS["rx_train_dir"], "train")
rx_test, rx_test_audit = load_rx_split(PATHS["rx_test_csv"], PATHS["rx_test_dir"], "test")
rx_all = pd.concat([rx_train, rx_test], ignore_index=True)

alias_path = CFG.DATA_ROOT / "merged_dataset" / "label_aliases.json"
with open(alias_path, "w", encoding="utf-8") as f:
    json.dump(rx_train_audit + rx_test_audit, f, indent=2)

print("Matched RxHandBD rows:", len(rx_all))
print("Unmatched audit rows:", sum(1 for item in rx_train_audit + rx_test_audit if item["matched_label"] is None))
print("Alias audit:", alias_path.resolve())
display(rx_all.head())

Matched RxHandBD rows: 560
Unmatched audit rows: 5018
Alias audit: D:\ediprjcursor\data\merged_dataset\label_aliases.json


,image_path,IMAGE,MEDICINE_NAME,GENERIC_NAME,source,source_split,raw_text,match_method,match_score
0,D:\ediprjcursor\data\RxHandBD-ML\Train_Set\P11...,P1138.jpg,Omastin,Fluconazole,RxHandBD-ML,train,Omastin,exact_clean,100.000000
1,D:\ediprjcursor\data\RxHandBD-ML\Train_Set\P11...,P1141.jpg,Denixil,Clonazepam,RxHandBD-ML,train,Deixil,difflib,92.307692
2,D:\ediprjcursor\data\RxHandBD-ML\Train_Set\P11...,P1153.jpg,Denixil,Clonazepam,RxHandBD-ML,train,Deixil,difflib,92.307692
3,D:\ediprjcursor\data\RxHandBD-ML\Train_Set\P11...,P1183.jpg,Nexum,Esomeprazole,RxHandBD-ML,train,Nexum,exact_clean,100.000000
4,D:\ediprjcursor\data\RxHandBD-ML\Train_Set\P11...,P1188.jpg,Denixil,Clonazepam,RxHandBD-ML,train,Deixil,difflib,92.307692


In [6]:
def existing_only(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["exists"] = out["image_path"].map(lambda p: Path(p).exists())
    missing = int((~out["exists"]).sum())
    if missing:
        print(f"Dropping {missing} rows with missing image files.")
    return out[out["exists"]].drop(columns=["exists"]).reset_index(drop=True)


baseline_rx = rx_train[rx_train["match_method"].eq("exact_clean")].copy()
baseline_df = existing_only(pd.concat([dh_train, baseline_rx], ignore_index=True))
expanded_df = existing_only(pd.concat([dh_all, rx_all], ignore_index=True))

baseline_path = CFG.DATA_ROOT / "merged_dataset" / "baseline_labels.csv"
expanded_path = CFG.DATA_ROOT / "merged_dataset" / "expanded_labels.csv"
baseline_df.to_csv(baseline_path, index=False)
expanded_df.to_csv(expanded_path, index=False)

encoder = LabelEncoder()
encoder.fit(canonical_labels)
with open(CFG.OUTPUTS_DIR / "label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)
with open(CFG.OUTPUTS_DIR / "generic_map.json", "w", encoding="utf-8") as f:
    json.dump(generic_map, f, indent=2, ensure_ascii=False)

print("baseline_df:", baseline_df.shape, "classes:", baseline_df["MEDICINE_NAME"].nunique())
print("expanded_df:", expanded_df.shape, "classes:", expanded_df["MEDICINE_NAME"].nunique())
print("Saved:", baseline_path.resolve(), expanded_path.resolve())

baseline_df: (3472, 9) classes: 78
expanded_df: (5240, 9) classes: 78
Saved: D:\ediprjcursor\data\merged_dataset\baseline_labels.csv D:\ediprjcursor\data\merged_dataset\expanded_labels.csv


In [7]:
def label_encode_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["label"] = encoder.transform(out["MEDICINE_NAME"])
    return out


def assert_all_classes(*frames: pd.DataFrame, label_col: str = "MEDICINE_NAME") -> None:
    expected = set(canonical_labels)
    for idx, frame in enumerate(frames):
        actual = set(frame[label_col].unique())
        missing = sorted(expected - actual)
        if missing:
            raise ValueError(f"Split {idx} is missing {len(missing)} classes: {missing[:10]}")


def split_80_10_10(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df, temp_df = train_test_split(
        df,
        test_size=0.20,
        stratify=df["MEDICINE_NAME"],
        random_state=CFG.RANDOM_STATE,
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        stratify=temp_df["MEDICINE_NAME"],
        random_state=CFG.RANDOM_STATE,
    )
    assert_all_classes(train_df, val_df, test_df)
    return map(label_encode_df, (train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)))


baseline_train_df, baseline_val_df, baseline_test_df = split_80_10_10(baseline_df)
expanded_train_df, expanded_val_df, expanded_test_df = split_80_10_10(expanded_df)

for name, frame in {
    "baseline_train": baseline_train_df,
    "baseline_val": baseline_val_df,
    "baseline_test": baseline_test_df,
    "expanded_train": expanded_train_df,
    "expanded_val": expanded_val_df,
    "expanded_test": expanded_test_df,
}.items():
    out_path = CFG.DATA_ROOT / "merged_dataset" / f"{name}.csv"
    frame.to_csv(out_path, index=False)
    print(name, frame.shape, "classes:", frame["MEDICINE_NAME"].nunique())

baseline_train (2777, 10) classes: 78
baseline_val (347, 10) classes: 78
baseline_test (348, 10) classes: 78
expanded_train (4192, 10) classes: 78
expanded_val (524, 10) classes: 78
expanded_test (524, 10) classes: 78


In [8]:
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path: tf.Tensor) -> tf.Tensor:
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image = tf.image.convert_image_dtype(image, tf.float32) * 255.0
    image.set_shape([None, None, 3])
    return image


def preprocess_direct(image: tf.Tensor) -> tf.Tensor:
    image = tf.image.resize(image, CFG.IMAGE_SIZE)
    return mobilenet_preprocess_input(image)


def preprocess_letterbox(image: tf.Tensor) -> tf.Tensor:
    image = tf.image.resize_with_pad(image, CFG.IMAGE_SIZE[0], CFG.IMAGE_SIZE[1])
    return mobilenet_preprocess_input(image)


def make_dataset(
    df: pd.DataFrame,
    preprocess_fn: Callable[[tf.Tensor], tf.Tensor] = preprocess_letterbox,
    shuffle: bool = False,
    cache: bool = True,
) -> tf.data.Dataset:
    paths = df["image_path"].astype(str).values
    labels = df["label"].astype(np.int32).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=CFG.RANDOM_STATE, reshuffle_each_iteration=True)

    def _load(path, label):
        image = load_image(path)
        image = preprocess_fn(image)
        return image, label

    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)
    if cache:
        ds = ds.cache()
    return ds.batch(CFG.BATCH_SIZE).prefetch(AUTOTUNE)


smoke_ds = make_dataset(expanded_train_df.head(32), shuffle=False, cache=False)
images, labels = next(iter(smoke_ds))
print(images.shape, labels.shape, images.dtype, labels.dtype)

(16, 224, 224, 3) (16,) <dtype: 'float32'> <dtype: 'int32'>


## Phase II: Model Building

In [9]:
def build_augmentation() -> keras.Sequential:
    return keras.Sequential(
        [
            layers.RandomRotation(0.03),
            layers.RandomTranslation(0.05, 0.05),
            layers.RandomZoom(0.05),
        ],
        name="handwriting_augmentation",
    )


def build_model(
    num_classes: int = CFG.NUM_CLASSES,
    augment: bool = True,
    trainable_base: bool = False,
) -> keras.Model:
    inputs = keras.Input(shape=(*CFG.IMAGE_SIZE, 3), name="image")
    x = build_augmentation()(inputs) if augment else inputs
    base = MobileNetV2(include_top=False, weights="imagenet", input_shape=(*CFG.IMAGE_SIZE, 3))
    base.trainable = trainable_base
    x = base(x, training=trainable_base)
    x = layers.GlobalAveragePooling2D(name="global_average_pool")(x)
    x = layers.Dropout(0.4, name="head_dropout_1")(x)
    x = layers.Dense(128, activation="relu", name="head_dense")(x)
    x = layers.Dropout(0.3, name="head_dropout_2")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="medicine")(x)
    return keras.Model(inputs, outputs, name="medicine_mobilenetv2")


def compile_model(model: keras.Model, learning_rate: float) -> keras.Model:
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


model_preview = compile_model(build_model(), CFG.LR_BASELINE)
model_preview.summary()

Model: "medicine_mobilenetv2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ handwriting_augmentation        │ (None, 224, 224, 3)    │             0 │
│ (Sequential)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pool             │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dropout_1 (Dropout)        │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dense (Dense)              │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dropout_2 (Dropout)        │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ medicine (Dense)                │ (None, 78)             │        10,062 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,432,014 (9.28 MB)

 Trainable params: 174,030 (679.80 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [10]:
def training_callbacks(checkpoint_path: Path) -> list[keras.callbacks.Callback]:
    return [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7),
        keras.callbacks.ModelCheckpoint(str(checkpoint_path), monitor="val_loss", save_best_only=True),
    ]


def evaluate_model(model: keras.Model, df: pd.DataFrame, ds: tf.data.Dataset, split_name: str) -> dict:
    probs = model.predict(ds, verbose=1)
    y_true = df["label"].to_numpy()
    y_pred = probs.argmax(axis=1)
    labels = np.arange(len(encoder.classes_))
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average="macro", zero_division=0
    )
    weighted_f1 = f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
    metrics = {
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "weighted_f1": weighted_f1,
    }
    print(pd.Series(metrics))
    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=encoder.classes_,
        zero_division=0,
        output_dict=True,
    )
    return {"metrics": metrics, "report": report, "y_true": y_true, "y_pred": y_pred, "probs": probs}


def top_confusion_pairs(y_true: np.ndarray, y_pred: np.ndarray, n: int = 20) -> pd.DataFrame:
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(encoder.classes_)))
    pairs = []
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if i != j and cm[i, j] > 0:
                pairs.append({
                    "true_label": encoder.classes_[i],
                    "predicted_label": encoder.classes_[j],
                    "count": int(cm[i, j]),
                })
    return pd.DataFrame(pairs).sort_values("count", ascending=False).head(n)

## Phase III: Experiment 1 - Baseline on 3,458 Sample Track

Training can take about an hour on CPU. Set `RUN_BASELINE_TRAINING = True` to execute this phase.

In [11]:
RUN_BASELINE_TRAINING = False

baseline_train_ds = make_dataset(baseline_train_df, preprocess_letterbox, shuffle=True)
baseline_val_ds = make_dataset(baseline_val_df, preprocess_letterbox)
baseline_test_ds = make_dataset(baseline_test_df, preprocess_letterbox)

if RUN_BASELINE_TRAINING:
    baseline_model = compile_model(build_model(augment=True, trainable_base=False), CFG.LR_BASELINE)
    baseline_history = baseline_model.fit(
        baseline_train_ds,
        validation_data=baseline_val_ds,
        epochs=CFG.EPOCHS,
        callbacks=training_callbacks(CFG.MODELS_DIR / "baseline_mobilenetv2.keras"),
    )
    baseline_eval = evaluate_model(baseline_model, baseline_test_df, baseline_test_ds, "baseline_test")
    print("Expected reference: validation about 63.01%, test about 59.83%.")
else:
    print("Baseline training skipped. Toggle RUN_BASELINE_TRAINING to run.")

Baseline training skipped. Toggle RUN_BASELINE_TRAINING to run.


## Phase IV: Experiment 2 - Production Model on Expanded Data

Set `RUN_PRODUCTION_TRAINING = True` when ready. The top 30 MobileNetV2 layers are unfrozen for fine-tuning.

In [12]:
RUN_PRODUCTION_TRAINING = False

expanded_train_ds = make_dataset(expanded_train_df, preprocess_letterbox, shuffle=True)
expanded_val_ds = make_dataset(expanded_val_df, preprocess_letterbox)
expanded_test_ds = make_dataset(expanded_test_df, preprocess_letterbox)

if RUN_PRODUCTION_TRAINING:
    production_model = compile_model(build_model(augment=True, trainable_base=False), CFG.LR_BASELINE)
    production_history_frozen = production_model.fit(
        expanded_train_ds,
        validation_data=expanded_val_ds,
        epochs=CFG.EPOCHS,
        callbacks=training_callbacks(CFG.MODELS_DIR / "production_mobilenetv2_frozen.keras"),
    )

    mobile_base = next(layer for layer in production_model.layers if isinstance(layer, keras.Model) and "mobilenet" in layer.name.lower())
    mobile_base.trainable = True
    for layer in mobile_base.layers[:-30]:
        layer.trainable = False

    compile_model(production_model, CFG.LR_FINETUNE)
    production_history_finetune = production_model.fit(
        expanded_train_ds,
        validation_data=expanded_val_ds,
        epochs=CFG.EPOCHS,
        callbacks=training_callbacks(CFG.MODELS_DIR / "production_mobilenetv2.keras"),
    )
    production_eval = evaluate_model(production_model, expanded_test_df, expanded_test_ds, "expanded_test")
else:
    print("Production training skipped. Toggle RUN_PRODUCTION_TRAINING to run.")

Production training skipped. Toggle RUN_PRODUCTION_TRAINING to run.


## Phase V: Additional Comparison Experiments

In [13]:
RUN_COMPARISON_EXPERIMENTS = False


def run_comparison_experiment(name: str, preprocess_fn, augment: bool) -> dict:
    train_ds = make_dataset(expanded_train_df, preprocess_fn, shuffle=True)
    val_ds = make_dataset(expanded_val_df, preprocess_fn)
    test_ds = make_dataset(expanded_test_df, preprocess_fn)
    model = compile_model(build_model(augment=augment, trainable_base=False), CFG.LR_BASELINE)
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG.COMPARE_EPOCHS,
        callbacks=training_callbacks(CFG.MODELS_DIR / f"{name}.keras"),
        verbose=1,
    )
    result = evaluate_model(model, expanded_test_df, test_ds, name)
    return result["metrics"]


if RUN_COMPARISON_EXPERIMENTS:
    comparison_rows = [
        run_comparison_experiment("exp3_direct_resize", preprocess_direct, True),
        run_comparison_experiment("exp3_letterbox", preprocess_letterbox, True),
        run_comparison_experiment("exp4_no_augmentation", preprocess_letterbox, False),
        run_comparison_experiment("exp4_with_augmentation", preprocess_letterbox, True),
    ]
    comparison_df = pd.DataFrame(comparison_rows)
    comparison_df.to_csv(CFG.OUTPUTS_DIR / "comparison_experiments.csv", index=False)
    display(comparison_df)
else:
    print("Comparison experiments skipped. Toggle RUN_COMPARISON_EXPERIMENTS to run.")

Comparison experiments skipped. Toggle RUN_COMPARISON_EXPERIMENTS to run.


## Phase VI: Error Analysis

In [14]:
def error_analysis(eval_result: dict, output_prefix: str = "production") -> tuple[pd.DataFrame, pd.DataFrame]:
    report_df = pd.DataFrame(eval_result["report"]).T
    class_rows = report_df.loc[encoder.classes_, ["precision", "recall", "f1-score", "support"]]
    bottom_10 = class_rows.sort_values("f1-score").head(10)
    confusions = top_confusion_pairs(eval_result["y_true"], eval_result["y_pred"], n=20)
    bottom_10.to_csv(CFG.OUTPUTS_DIR / f"{output_prefix}_bottom10_f1.csv")
    confusions.to_csv(CFG.OUTPUTS_DIR / f"{output_prefix}_top_confusions.csv", index=False)
    display(bottom_10)
    display(confusions)
    return bottom_10, confusions


print("After running production evaluation, call: error_analysis(production_eval)")
print("Research gaps: limited data, handwriting variability, similar names, domain shift, and class imbalance.")

After running production evaluation, call: error_analysis(production_eval)
Research gaps: limited data, handwriting variability, similar names, domain shift, and class imbalance.


## Phase VII: Visualizations and Inference

In [15]:
def plot_history(history, output_name: str) -> None:
    hist = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    hist[["accuracy", "val_accuracy"]].plot(ax=axes[0], title="Train/Validation Accuracy")
    hist[["loss", "val_loss"]].plot(ax=axes[1], title="Train/Validation Loss")
    fig.tight_layout()
    fig.savefig(CFG.OUTPUTS_DIR / output_name, dpi=180)
    plt.show()


def plot_confusion_heatmap(y_true, y_pred, output_name: str = "confusion_heatmap.png") -> None:
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(encoder.classes_)))
    plt.figure(figsize=(18, 15))
    sns.heatmap(cm, cmap="Blues", xticklabels=encoder.classes_, yticklabels=encoder.classes_)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(CFG.OUTPUTS_DIR / output_name, dpi=180)
    plt.show()


def plot_confidence_histogram(probs: np.ndarray, output_name: str = "confidence_histogram.png") -> None:
    confidence = probs.max(axis=1)
    plt.figure(figsize=(8, 4))
    sns.histplot(confidence, bins=30)
    plt.xlabel("Top prediction confidence")
    plt.tight_layout()
    plt.savefig(CFG.OUTPUTS_DIR / output_name, dpi=180)
    plt.show()


def show_examples(df: pd.DataFrame, eval_result: dict, correct: bool = True, n: int = 10) -> None:
    mask = eval_result["y_true"] == eval_result["y_pred"]
    idxs = np.where(mask if correct else ~mask)[0][:n]
    cols = min(5, len(idxs))
    rows = math.ceil(len(idxs) / cols) if cols else 1
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 2.5 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, idx in zip(axes, idxs):
        image = Image.open(df.iloc[idx]["image_path"])
        true_label = encoder.inverse_transform([eval_result["y_true"][idx]])[0]
        pred_label = encoder.inverse_transform([eval_result["y_pred"][idx]])[0]
        ax.imshow(image, cmap="gray")
        ax.set_title(f"T: {true_label}\nP: {pred_label}", fontsize=8)
        ax.axis("off")
    for ax in axes[len(idxs):]:
        ax.axis("off")
    plt.tight_layout()
    suffix = "correct" if correct else "incorrect"
    plt.savefig(CFG.OUTPUTS_DIR / f"examples_{suffix}.png", dpi=180)
    plt.show()

In [16]:
DISCLAIMER = (
    "Research prototype only. Verify every prediction against the original prescription; "
    "this is not medical advice and does not recommend dosage or treatment."
)


def load_inference_assets(model_path: Path = CFG.MODELS_DIR / "production_mobilenetv2.keras"):
    model = keras.models.load_model(model_path)
    with open(CFG.OUTPUTS_DIR / "label_encoder.pkl", "rb") as f:
        label_encoder = pickle.load(f)
    with open(CFG.OUTPUTS_DIR / "generic_map.json", "r", encoding="utf-8") as f:
        med_to_generic = json.load(f)
    return model, label_encoder, med_to_generic


def preprocess_image_for_prediction(image_path: str | Path) -> tf.Tensor:
    image = load_image(tf.convert_to_tensor(str(image_path)))
    image = preprocess_letterbox(image)
    return tf.expand_dims(image, axis=0)


def predict_medicine(
    image_path: str | Path,
    model: keras.Model | None = None,
    label_encoder: LabelEncoder | None = None,
    med_to_generic: dict | None = None,
    threshold: float = CFG.CONFIDENCE_THRESHOLD,
    model_path: Path = CFG.MODELS_DIR / "production_mobilenetv2.keras",
) -> dict:
    if model is None or label_encoder is None or med_to_generic is None:
        model, label_encoder, med_to_generic = load_inference_assets(model_path)

    probs = model.predict(preprocess_image_for_prediction(image_path), verbose=0)[0]
    top_idx = int(np.argmax(probs))
    top_label = str(label_encoder.inverse_transform([top_idx])[0])
    top5_idx = probs.argsort()[-5:][::-1]
    top5 = {str(label_encoder.inverse_transform([int(i)])[0]): float(probs[i]) for i in top5_idx}
    confidence = float(probs[top_idx])
    return {
        "predicted_medicine_name": top_label,
        "predicted_generic_name": med_to_generic.get(top_label, ""),
        "confidence": confidence,
        "top_5_predictions": top5,
        "is_confident": confidence >= threshold,
        "warning": None if confidence >= threshold else "Low confidence: verify carefully against prescription.",
        "disclaimer": DISCLAIMER,
    }


print("After training/saving production model, call predict_medicine('../data/.../image.png').")

After training/saving production model, call predict_medicine('../data/.../image.png').


## Phase VIII: Interactive Feedback and Continuous Learning

In [17]:
FEEDBACK_COLUMNS = ["image_path", "correct_label", "predicted_label", "confidence", "timestamp"]
FEEDBACK_CSV = CFG.FEEDBACK_DIR / "feedback_corrections.csv"


def ensure_feedback_csv() -> None:
    if not FEEDBACK_CSV.exists():
        pd.DataFrame(columns=FEEDBACK_COLUMNS).to_csv(FEEDBACK_CSV, index=False)


def log_feedback(image_path: str | Path, correct_label: str, predicted_label: str, confidence: float) -> None:
    ensure_feedback_csv()
    if correct_label not in set(encoder.classes_):
        raise ValueError(f"correct_label must be one of the 78 canonical labels.")
    row = {
        "image_path": str(Path(image_path).resolve()),
        "correct_label": correct_label,
        "predicted_label": predicted_label,
        "confidence": float(confidence),
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    }
    pd.DataFrame([row]).to_csv(FEEDBACK_CSV, mode="a", header=False, index=False)
    print(f"Logged correction to {FEEDBACK_CSV.resolve()}")


ensure_feedback_csv()
print(FEEDBACK_CSV.resolve())

D:\ediprjcursor\data\feedback\feedback_corrections.csv


In [18]:
def interactive_correction(image_path: str | Path) -> dict:
    model, label_encoder, med_to_generic = load_inference_assets()
    result = predict_medicine(image_path, model, label_encoder, med_to_generic)
    display(DisplayImage(filename=str(image_path)))
    print(json.dumps(result, indent=2))
    answer = input("Is the top prediction correct? (y/n): ").strip().lower()
    if answer == "n":
        correct_label = input("Enter correct canonical medicine name: ").strip()
        if correct_label not in set(label_encoder.classes_):
            raise ValueError("Unknown medicine name. Use one of the canonical 78 labels.")
        log_feedback(image_path, correct_label, result["predicted_medicine_name"], result["confidence"])
    return result


print("Call interactive_correction(image_path) after a production model has been saved.")

Call interactive_correction(image_path) after a production model has been saved.


In [19]:
def next_model_version() -> int:
    existing = sorted(CFG.MODELS_DIR.glob("production_mobilenetv2_v*.keras"))
    versions = []
    for path in existing:
        match = re.search(r"_v(\d+)\.keras$", path.name)
        if match:
            versions.append(int(match.group(1)))
    return max(versions, default=0) + 1


def build_feedback_dataframe(feedback_df: pd.DataFrame) -> pd.DataFrame:
    out = feedback_df.copy()
    out["MEDICINE_NAME"] = out["correct_label"]
    out["GENERIC_NAME"] = out["correct_label"].map(generic_map).fillna("")
    out["source"] = "feedback"
    out["source_split"] = "feedback"
    out["IMAGE"] = out["image_path"].map(lambda p: Path(p).name)
    out["label"] = encoder.transform(out["MEDICINE_NAME"])
    return out[["image_path", "IMAGE", "MEDICINE_NAME", "GENERIC_NAME", "source", "source_split", "label"]]


def update_model_with_feedback(batch_size: int = CFG.FEEDBACK_BATCH_SIZE, mix_ratio: float = 0.5, epochs: int = 5) -> Path | None:
    ensure_feedback_csv()
    feedback = pd.read_csv(FEEDBACK_CSV)
    if len(feedback) < batch_size:
        print(f"Need at least {batch_size} corrections; found {len(feedback)}.")
        return None

    model_path = CFG.MODELS_DIR / "production_mobilenetv2.keras"
    if not model_path.exists():
        raise FileNotFoundError(f"Train or place a production model at {model_path.resolve()} first.")

    feedback_batch = feedback.head(batch_size)
    feedback_train = build_feedback_dataframe(feedback_batch)
    replay_n = max(batch_size, int(batch_size * mix_ratio / max(1e-9, 1 - mix_ratio)))
    replay = expanded_train_df.sample(n=min(replay_n, len(expanded_train_df)), random_state=CFG.RANDOM_STATE)
    mixed = pd.concat([feedback_train, replay], ignore_index=True)
    train_ds = make_dataset(mixed, preprocess_letterbox, shuffle=True, cache=False)

    model = keras.models.load_model(model_path)
    for layer in model.layers:
        if isinstance(layer, keras.Model) and "mobilenet" in layer.name.lower():
            layer.trainable = True
            for sublayer in layer.layers[:-30]:
                sublayer.trainable = False
    compile_model(model, CFG.LR_FEEDBACK)
    model.fit(train_ds, epochs=epochs)

    version = next_model_version()
    updated_path = CFG.MODELS_DIR / f"production_mobilenetv2_v{version}.keras"
    model.save(updated_path)

    remaining = feedback.iloc[batch_size:].copy()
    archive_path = CFG.FEEDBACK_DIR / f"processed_feedback_v{version}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    feedback_batch.to_csv(archive_path, index=False)
    remaining.to_csv(FEEDBACK_CSV, index=False)

    metadata = {
        "version": version,
        "model_path": str(updated_path),
        "processed_rows": int(len(feedback_batch)),
        "archive_path": str(archive_path),
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    }
    with open(CFG.FEEDBACK_DIR / f"update_metadata_v{version}.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    before = keras.models.load_model(model_path)
    before_eval = evaluate_model(before, expanded_test_df, expanded_test_ds, f"production_before_v{version}")
    after_eval = evaluate_model(model, expanded_test_df, expanded_test_ds, f"production_after_v{version}")
    print("Accuracy delta:", after_eval["metrics"]["accuracy"] - before_eval["metrics"]["accuracy"])
    print("Saved updated model:", updated_path.resolve())
    return updated_path

### Immediate Single-Sample Update

This is for demonstration only. Single-sample updates are slower and less stable than batch updates with replay data.

In [20]:
def update_model_on_sample(image_path: str | Path, correct_label: str, epochs: int = 1) -> Path:
    if correct_label not in set(encoder.classes_):
        raise ValueError("correct_label must be one of the canonical 78 labels.")
    model_path = CFG.MODELS_DIR / "production_mobilenetv2.keras"
    if not model_path.exists():
        raise FileNotFoundError(f"Train or place a production model at {model_path.resolve()} first.")

    one = pd.DataFrame([{
        "image_path": str(Path(image_path).resolve()),
        "IMAGE": Path(image_path).name,
        "MEDICINE_NAME": correct_label,
        "GENERIC_NAME": generic_map.get(correct_label, ""),
        "source": "feedback_demo",
        "source_split": "single",
        "label": int(encoder.transform([correct_label])[0]),
    }])
    ds = make_dataset(one, preprocess_letterbox, shuffle=True, cache=False)
    model = keras.models.load_model(model_path)
    compile_model(model, CFG.LR_FEEDBACK)
    model.fit(ds, epochs=epochs)
    updated_path = CFG.MODELS_DIR / f"production_mobilenetv2_single_{datetime.now().strftime('%Y%m%d_%H%M%S')}.keras"
    model.save(updated_path)
    print("Saved demo single-sample update:", updated_path.resolve())
    return updated_path

## Deliverable Checklist

- `data/merged_dataset/baseline_labels.csv`
- `data/merged_dataset/expanded_labels.csv`
- `data/merged_dataset/label_aliases.json`
- `outputs/label_encoder.pkl`
- `outputs/generic_map.json`
- `models/baseline_mobilenetv2.keras` after Phase III training
- `models/production_mobilenetv2.keras` after Phase IV training
- `data/feedback/feedback_corrections.csv`
- evaluation plots and CSVs under `outputs/`